In [1]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore") 
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils
from sklearn.cluster import KMeans

In [2]:
def calculate_bic(bn, data):
    """
    Calculates BIC score for a CausalNex BayesianNetwork.
    Formula: BIC = k * ln(n) - 2 * ln(L)
    Returns: bic, log_likelihood
    """
    log_likelihood = 0
    k = 0
    n = len(data)

    for node in bn.nodes:
        cpd = bn.cpds[node]
        
        probs = bn.predict_probability(data, node)
        
        true_values = data[node]
        
        y_dummies = pd.get_dummies(true_values)
        
        y_dummies = y_dummies.reindex(columns=probs.columns, fill_value=0)
        
        p_actual = (probs * y_dummies).sum(axis=1)
        
        log_likelihood += np.sum(np.log(p_actual + 1e-9))

        num_parent_configs = cpd.shape[0]
        num_node_states = cpd.shape[1]
        k += num_parent_configs * (num_node_states - 1)

    bic = k * np.log(n) - 2 * log_likelihood
    
    return bic, log_likelihood

In [3]:
from sklearn.mixture import GaussianMixture # <--- NEW IMPORT
from sklearn.preprocessing import StandardScaler
import itertools

def add_latent_variable_gmm(bn_baseline, sm_baseline, children_cols, discretised_data, raw_data=None):
    
    if raw_data is None:
        raise ValueError("You must provide the continuous 'raw_data' to accurately estimate the Latent Variable.")
        
    subset_data = raw_data[children_cols].loc[discretised_data.index].copy()
    
    
    print(f"Estimating LV based on continuous GMM distributions in: {children_cols}")

    scaler = StandardScaler()
    subset_data_scaled = scaler.fit_transform(subset_data)
    
    gmm = GaussianMixture(n_components=2, random_state=42, n_init=10)
    proxy_lv = gmm.fit_predict(subset_data_scaled)
    
    discretised_with_lv = discretised_data.copy()
    discretised_with_lv['LV'] = proxy_lv
    discretised_with_lv['LV'] = discretised_with_lv['LV'].map({0: "State0", 1: "State1"})
    
    sm_with_lv = sm_baseline.copy()
    sm_with_lv.add_node('LV')
    
    for child in children_cols:
        sm_with_lv.add_edge('LV', child)
        
    for u, v in itertools.permutations(children_cols, 2):
        if sm_with_lv.has_edge(u, v):
            print(f"  - Removing direct edge: {u} -> {v}")
            sm_with_lv.remove_edge(u, v)

    bn_with_lv = BayesianNetwork(sm_with_lv)
    bn_with_lv.fit_node_states(discretised_with_lv)
    bn_with_lv.fit_cpds(discretised_with_lv, method="BayesianEstimator", bayes_prior="K2")
    
    bic_lv, ll_lv = calculate_bic(bn_with_lv, discretised_with_lv)
    print(f"LV Model BIC: {bic_lv:.2f} (LogLikelihood: {ll_lv:.2f})")
    
    return bn_with_lv, discretised_with_lv, bic_lv

In [21]:
df_lv = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lv_bn_no_context.csv")

df_lv = df_lv.drop(["Unnamed: 0"], axis = 1)

df_lv

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,9775.522707,22.75,71.978571,5.390000,3.110000,10.56,29.615,-20.108026,18.600884,0.134663,9.22,-7.7,21.2,0.000004,63.181818,44.685377
1,10725.051692,22.84,71.978571,5.590000,2.850000,13.65,29.811,-13.299099,20.701069,-3.172711,4.20,-13.3,25.9,0.000004,94.495413,69.408798
2,8629.945617,21.52,66.300000,5.480000,3.190000,33.43,34.317,-2.039931,19.573897,-7.966839,-0.58,-19.8,26.4,0.000004,112.149533,198.581136
3,8447.759415,20.86,63.900000,5.440000,3.110000,36.24,30.375,-2.131443,18.789970,-8.375610,2.02,-21.9,20.9,0.000004,84.761905,220.935379
4,9281.534430,21.42,66.100000,5.170000,3.100000,31.07,33.478,-6.131271,19.255704,-3.827185,5.72,-14.5,19.0,0.000004,83.980583,248.557865
5,10374.147811,21.42,67.900000,5.270000,3.160000,28.56,35.709,-5.548474,17.988056,-1.006694,3.99,-10.3,19.2,0.000004,95.073892,351.810791
6,10909.112726,20.82,69.500000,5.390000,3.000000,23.24,37.037,-4.525224,18.281111,-0.749488,3.39,-11.1,19.4,0.000004,111.940299,310.275825
7,11430.790829,19.62,70.600000,5.350000,3.090000,19.59,38.629,-2.898049,18.488335,-1.357478,3.45,-7.7,21.2,0.000004,51.758794,358.109379
8,12007.048703,19.78,72.500000,5.370000,3.310000,16.32,37.538,-1.188889,18.709564,-1.740458,3.09,-8.7,22.5,0.000004,68.686869,304.485269
9,12502.021651,19.17,73.000000,5.100000,3.200000,17.27,37.138,0.526321,18.476344,-0.184948,1.24,-9.6,21.8,0.000004,48.469388,79.129692


In [22]:
lv = pd.DataFrame()
lv = df_lv.copy()
lv['VABI_1'] = lv['VABI'].shift(1)
lv['YUA_2'] = lv['YUA'].shift(2)

In [23]:
lv = lv.dropna()

In [24]:
discretised_lv = pd.DataFrame(index=lv.index)

for col in lv.columns:
    no_unique = lv[col].nunique()
    
    if no_unique <= 1:
        discretised_lv[col] = 0
    else:
        try:
            discretised_lv[col] = pd.qcut(
                lv[col], 
                q=min(2, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_lv[col] = lv[col].rank(method='lvnse').astype(int) - 1
    num_states = discretised_lv[col].nunique()
    
    if num_states == 2:
        discretised_lv[col] = discretised_lv[col].map({0: "Low", 1: "High"})
    elif num_states == 3:
        discretised_lv[col] = discretised_lv[col].map({0: "Low", 1: "Medium", 2: "High"})
    else:
        discretised_lv[col] = discretised_lv[col].astype(str)

print("Discretised data:")
print(discretised_lv)

Discretised data:
    GDPC  VABI   EMP   QOA   QOR   YUA  REST  EBSG  GGFC  LBGD   FDI    PC  \
2    Low  High   Low  High  High  High   Low  High  High   Low   Low   Low   
3    Low  High   Low  High   Low  High   Low   Low  High   Low   Low   Low   
4    Low  High   Low   Low   Low  High   Low   Low  High   Low  High   Low   
5    Low  High   Low   Low   Low  High   Low   Low   Low  High  High   Low   
6    Low  High   Low  High   Low  High   Low   Low   Low  High  High   Low   
7    Low   Low   Low   Low   Low  High  High   Low   Low  High  High  High   
8    Low   Low   Low   Low  High   Low   Low  High   Low   Low   Low  High   
9   High   Low  High   Low  High  High   Low  High   Low  High   Low   Low   
10  High   Low  High   Low   Low   Low  High  High   Low  High  High  High   
11  High   Low  High  High  High   Low  High  High   Low  High   Low  High   
12  High   Low  High  High  High   Low  High  High  High  High  High  High   
13  High   Low  High   Low   Low   Low  High  

In [25]:
sm = StructureModel()

In [26]:
sm.add_edges_from([  
  ('EUPC', 'QOA'),
    ('VABI', 'QOA'),
    ('EUPC', 'REST'),
    ('EUPC', 'EBSG'),
    ('VABI_1', 'EBSG'),
    ('EBSG', 'LBGD'),
    ('EBSG', 'FDI'),
    ('VABI', 'FDI'),
    ('EBSG', 'PC'),
    ('LBGD', 'ARP'),
    ('YUA_2', 'EUPC'),
    ('REST', 'VABI')
])



In [27]:
sm.edges

OutEdgeView([('EUPC', 'QOA'), ('EUPC', 'REST'), ('EUPC', 'EBSG'), ('VABI', 'QOA'), ('VABI', 'FDI'), ('REST', 'VABI'), ('EBSG', 'LBGD'), ('EBSG', 'FDI'), ('EBSG', 'PC'), ('VABI_1', 'EBSG'), ('LBGD', 'ARP'), ('YUA_2', 'EUPC')])

In [28]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_1.html")

Graphs/fully_connected_dk_1.html


In [34]:
bn = BayesianNetwork(sm)

bn.fit_node_states(discretised_lv)

bn.fit_cpds(discretised_lv, method="BayesianEstimator", bayes_prior="K2")

bic_baseline, ll_baseline = calculate_bic(bn, discretised_lv)
print(f"Baseline BIC: {bic_baseline:.2f} (LogLikelihood: {ll_baseline:.2f})")

Baseline BIC: 6461.94 (LogLikelihood: -3191.38)


In [35]:
bn_with_lv_1, data_with_lv_1, bic_lv_1 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['EUPC', 'QOA'],
    discretised_data=discretised_lv,
    raw_data=lv
)

Estimating LV based on continuous GMM distributions in: ['EUPC', 'QOA']
  - Removing direct edge: EUPC -> QOA
LV Model BIC: 7052.75 (LogLikelihood: -3481.51)


In [36]:
bn_with_lv_2, data_with_lv_2, bic_lv_2 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['EUPC', 'REST'], 
    discretised_data=discretised_lv,
    raw_data=lv
)

Estimating LV based on continuous GMM distributions in: ['EUPC', 'REST']
  - Removing direct edge: EUPC -> REST
LV Model BIC: 7052.75 (LogLikelihood: -3481.51)


In [37]:
bn_with_lv_3, data_with_lv_3, bic_lv_3 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['EUPC', 'EBSG'], 
    discretised_data=discretised_lv,
    raw_data=lv
)

Estimating LV based on continuous GMM distributions in: ['EUPC', 'EBSG']
  - Removing direct edge: EUPC -> EBSG
LV Model BIC: 7052.75 (LogLikelihood: -3481.51)


In [38]:
bn_with_lv_4, data_with_lv_4, bic_lv_4 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['YUA_2', 'EUPC'], 
    discretised_data=discretised_lv,
    raw_data=lv
)

Estimating LV based on continuous GMM distributions in: ['YUA_2', 'EUPC']
  - Removing direct edge: YUA_2 -> EUPC
LV Model BIC: 7047.47 (LogLikelihood: -3481.51)


In [14]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore") 
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils
from sklearn.cluster import KMeans

In [15]:
df_lv = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lv_bn_no_context.csv")

df_lv = df_lv.drop(["Unnamed: 0"], axis = 1)

df_lv

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,9775.522707,22.75,71.978571,5.390000,3.110000,10.56,29.615,-20.108026,18.600884,0.134663,9.22,-7.7,21.2,0.000004,63.181818,44.685377
1,10725.051692,22.84,71.978571,5.590000,2.850000,13.65,29.811,-13.299099,20.701069,-3.172711,4.20,-13.3,25.9,0.000004,94.495413,69.408798
2,8629.945617,21.52,66.300000,5.480000,3.190000,33.43,34.317,-2.039931,19.573897,-7.966839,-0.58,-19.8,26.4,0.000004,112.149533,198.581136
3,8447.759415,20.86,63.900000,5.440000,3.110000,36.24,30.375,-2.131443,18.789970,-8.375610,2.02,-21.9,20.9,0.000004,84.761905,220.935379
4,9281.534430,21.42,66.100000,5.170000,3.100000,31.07,33.478,-6.131271,19.255704,-3.827185,5.72,-14.5,19.0,0.000004,83.980583,248.557865
5,10374.147811,21.42,67.900000,5.270000,3.160000,28.56,35.709,-5.548474,17.988056,-1.006694,3.99,-10.3,19.2,0.000004,95.073892,351.810791
6,10909.112726,20.82,69.500000,5.390000,3.000000,23.24,37.037,-4.525224,18.281111,-0.749488,3.39,-11.1,19.4,0.000004,111.940299,310.275825
7,11430.790829,19.62,70.600000,5.350000,3.090000,19.59,38.629,-2.898049,18.488335,-1.357478,3.45,-7.7,21.2,0.000004,51.758794,358.109379
8,12007.048703,19.78,72.500000,5.370000,3.310000,16.32,37.538,-1.188889,18.709564,-1.740458,3.09,-8.7,22.5,0.000004,68.686869,304.485269
9,12502.021651,19.17,73.000000,5.100000,3.200000,17.27,37.138,0.526321,18.476344,-0.184948,1.24,-9.6,21.8,0.000004,48.469388,79.129692


In [16]:
lv = pd.DataFrame()
lv = df_lv.copy()
lv['EUPC_1'] = lv['EUPC'].shift(1)
lv['CO2C_2'] = lv['CO2C'].shift(2)

In [17]:
lv = lv.dropna()

In [18]:
discretised_lv = pd.DataFrame(index=lv.index)

for col in lv.columns:
    no_unique = lv[col].nunique()
    
    if no_unique <= 1:
        discretised_lv[col] = 0
    else:
        try:
            discretised_lv[col] = pd.qcut(
                lv[col], 
                q=min(2, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_lv[col] = lv[col].rank(method='lvnse').astype(int) - 1
    num_states = discretised_lv[col].nunique()
    
    if num_states == 2:
        discretised_lv[col] = discretised_lv[col].map({0: "Low", 1: "High"})
    elif num_states == 3:
        discretised_lv[col] = discretised_lv[col].map({0: "Low", 1: "Medium", 2: "High"})
    else:
        discretised_lv[col] = discretised_lv[col].astype(str)

print("Discretised data:")
print(discretised_lv)

Discretised data:
    GDPC  VABI   EMP   QOA   QOR   YUA  REST  EBSG  GGFC  LBGD   FDI    PC  \
2    Low  High   Low  High  High  High   Low  High  High   Low   Low   Low   
3    Low  High   Low  High   Low  High   Low   Low  High   Low   Low   Low   
4    Low  High   Low   Low   Low  High   Low   Low  High   Low  High   Low   
5    Low  High   Low   Low   Low  High   Low   Low   Low  High  High   Low   
6    Low  High   Low  High   Low  High   Low   Low   Low  High  High   Low   
7    Low   Low   Low   Low   Low  High  High   Low   Low  High  High  High   
8    Low   Low   Low   Low  High   Low   Low  High   Low   Low   Low  High   
9   High   Low  High   Low  High  High   Low  High   Low  High   Low   Low   
10  High   Low  High   Low   Low   Low  High  High   Low  High  High  High   
11  High   Low  High  High  High   Low  High  High   Low  High   Low  High   
12  High   Low  High  High  High   Low  High  High  High  High  High  High   
13  High   Low  High   Low   Low   Low  High  

In [19]:
sm = StructureModel()

In [20]:
sm.add_edges_from([
    ('EUPC_1', 'QOR'),  
     ('CO2C_2', 'QOR')
])

In [21]:
sm.edges

OutEdgeView([('EUPC_1', 'QOR'), ('CO2C_2', 'QOR')])

In [22]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_2.html")

Graphs/fully_connected_dk_2.html


In [23]:
bn = BayesianNetwork(sm)

bn.fit_node_states(discretised_lv)

bn.fit_cpds(discretised_lv, method="BayesianEstimator", bayes_prior="K2")

bic_baseline, ll_baseline = calculate_bic(bn, discretised_lv)
print(f"Baseline BIC: {bic_baseline:.2f} (LogLikelihood: {ll_baseline:.2f})")

Baseline BIC: 1756.59 (LogLikelihood: -870.38)


In [24]:
bn_with_lv, data_with_lv, bic_lv = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['EUPC_1', 'QOR'],
    discretised_data=discretised_lv,
    raw_data=lv
)

Estimating LV based on continuous GMM distributions in: ['EUPC_1', 'QOR']
  - Removing direct edge: EUPC_1 -> QOR
LV Model BIC: 2342.12 (LogLikelihood: -1160.50)


In [25]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore") 
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils
from sklearn.cluster import KMeans

In [26]:
df_lv = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lv_bn_no_context.csv")

df_lv = df_lv.drop(["Unnamed: 0"], axis = 1)

df_lv

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,9775.522707,22.75,71.978571,5.390000,3.110000,10.56,29.615,-20.108026,18.600884,0.134663,9.22,-7.7,21.2,0.000004,63.181818,44.685377
1,10725.051692,22.84,71.978571,5.590000,2.850000,13.65,29.811,-13.299099,20.701069,-3.172711,4.20,-13.3,25.9,0.000004,94.495413,69.408798
2,8629.945617,21.52,66.300000,5.480000,3.190000,33.43,34.317,-2.039931,19.573897,-7.966839,-0.58,-19.8,26.4,0.000004,112.149533,198.581136
3,8447.759415,20.86,63.900000,5.440000,3.110000,36.24,30.375,-2.131443,18.789970,-8.375610,2.02,-21.9,20.9,0.000004,84.761905,220.935379
4,9281.534430,21.42,66.100000,5.170000,3.100000,31.07,33.478,-6.131271,19.255704,-3.827185,5.72,-14.5,19.0,0.000004,83.980583,248.557865
5,10374.147811,21.42,67.900000,5.270000,3.160000,28.56,35.709,-5.548474,17.988056,-1.006694,3.99,-10.3,19.2,0.000004,95.073892,351.810791
6,10909.112726,20.82,69.500000,5.390000,3.000000,23.24,37.037,-4.525224,18.281111,-0.749488,3.39,-11.1,19.4,0.000004,111.940299,310.275825
7,11430.790829,19.62,70.600000,5.350000,3.090000,19.59,38.629,-2.898049,18.488335,-1.357478,3.45,-7.7,21.2,0.000004,51.758794,358.109379
8,12007.048703,19.78,72.500000,5.370000,3.310000,16.32,37.538,-1.188889,18.709564,-1.740458,3.09,-8.7,22.5,0.000004,68.686869,304.485269
9,12502.021651,19.17,73.000000,5.100000,3.200000,17.27,37.138,0.526321,18.476344,-0.184948,1.24,-9.6,21.8,0.000004,48.469388,79.129692


In [27]:
lv = pd.DataFrame()
lv = df_lv.copy()
lv['VABI_1'] = lv['VABI'].shift(1)
lv['YUA_2'] = lv['YUA'].shift(2)

In [28]:
lv = lv.dropna()

In [29]:
discretised_lv = pd.DataFrame(index=lv.index)

for col in lv.columns:
    no_unique = lv[col].nunique()
    
    if no_unique <= 1:
        discretised_lv[col] = 0
    else:
        try:
            discretised_lv[col] = pd.qcut(
                lv[col], 
                q=min(2, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_lv[col] = lv[col].rank(method='lvnse').astype(int) - 1
    num_states = discretised_lv[col].nunique()
    
    if num_states == 2:
        discretised_lv[col] = discretised_lv[col].map({0: "Low", 1: "High"})
    elif num_states == 3:
        discretised_lv[col] = discretised_lv[col].map({0: "Low", 1: "Medium", 2: "High"})
    else:
        discretised_lv[col] = discretised_lv[col].astype(str)

print("Discretised data:")
print(discretised_lv)

Discretised data:
    GDPC  VABI   EMP   QOA   QOR   YUA  REST  EBSG  GGFC  LBGD   FDI    PC  \
2    Low  High   Low  High  High  High   Low  High  High   Low   Low   Low   
3    Low  High   Low  High   Low  High   Low   Low  High   Low   Low   Low   
4    Low  High   Low   Low   Low  High   Low   Low  High   Low  High   Low   
5    Low  High   Low   Low   Low  High   Low   Low   Low  High  High   Low   
6    Low  High   Low  High   Low  High   Low   Low   Low  High  High   Low   
7    Low   Low   Low   Low   Low  High  High   Low   Low  High  High  High   
8    Low   Low   Low   Low  High   Low   Low  High   Low   Low   Low  High   
9   High   Low  High   Low  High  High   Low  High   Low  High   Low   Low   
10  High   Low  High   Low   Low   Low  High  High   Low  High  High  High   
11  High   Low  High  High  High   Low  High  High   Low  High   Low  High   
12  High   Low  High  High  High   Low  High  High  High  High  High  High   
13  High   Low  High   Low   Low   Low  High  

In [30]:
sm = StructureModel()

In [31]:
sm.add_edges_from([
     ('QOA', 'EUPC'),
    ('VABI', 'QOA'),
    ('EUPC', 'EBSG'),
    ('VABI_1', 'EBSG'),
    ('EBSG', 'LBGD'),
    ('EBSG', 'FDI'),
    ('VABI', 'FDI'),
    ('EBSG', 'PC'),
    ('LBGD', 'ARP'),
    ('YUA_2', 'EUPC'),
    ('REST', 'VABI')
])

In [32]:
sm.edges

OutEdgeView([('QOA', 'EUPC'), ('EUPC', 'EBSG'), ('VABI', 'QOA'), ('VABI', 'FDI'), ('EBSG', 'LBGD'), ('EBSG', 'FDI'), ('EBSG', 'PC'), ('VABI_1', 'EBSG'), ('LBGD', 'ARP'), ('YUA_2', 'EUPC'), ('REST', 'VABI')])

In [33]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_2.html")

Graphs/fully_connected_dk_2.html


In [34]:
bn = BayesianNetwork(sm)

bn.fit_node_states(discretised_lv)

bn.fit_cpds(discretised_lv, method="BayesianEstimator", bayes_prior="K2")

bic_baseline, ll_baseline = calculate_bic(bn, discretised_lv)
print(f"Baseline BIC: {bic_baseline:.2f} (LogLikelihood: {ll_baseline:.2f})")

Baseline BIC: 6456.66 (LogLikelihood: -3191.38)


In [35]:
bn_with_lv_5, data_with_lv_5, bic_lv_5 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['QOA', 'EUPC'],
    discretised_data=discretised_lv,
    raw_data=lv
)

Estimating LV based on continuous GMM distributions in: ['QOA', 'EUPC']
  - Removing direct edge: QOA -> EUPC
LV Model BIC: 7047.47 (LogLikelihood: -3481.51)


In [4]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore") 
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils
from sklearn.cluster import KMeans

In [5]:
df_lv = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lv_bn_no_context.csv")

df_lv = df_lv.drop(["Unnamed: 0"], axis = 1)

df_lv

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,9775.522707,22.75,71.978571,5.390000,3.110000,10.56,29.615,-20.108026,18.600884,0.134663,9.22,-7.7,21.2,0.000004,63.181818,44.685377
1,10725.051692,22.84,71.978571,5.590000,2.850000,13.65,29.811,-13.299099,20.701069,-3.172711,4.20,-13.3,25.9,0.000004,94.495413,69.408798
2,8629.945617,21.52,66.300000,5.480000,3.190000,33.43,34.317,-2.039931,19.573897,-7.966839,-0.58,-19.8,26.4,0.000004,112.149533,198.581136
3,8447.759415,20.86,63.900000,5.440000,3.110000,36.24,30.375,-2.131443,18.789970,-8.375610,2.02,-21.9,20.9,0.000004,84.761905,220.935379
4,9281.534430,21.42,66.100000,5.170000,3.100000,31.07,33.478,-6.131271,19.255704,-3.827185,5.72,-14.5,19.0,0.000004,83.980583,248.557865
5,10374.147811,21.42,67.900000,5.270000,3.160000,28.56,35.709,-5.548474,17.988056,-1.006694,3.99,-10.3,19.2,0.000004,95.073892,351.810791
6,10909.112726,20.82,69.500000,5.390000,3.000000,23.24,37.037,-4.525224,18.281111,-0.749488,3.39,-11.1,19.4,0.000004,111.940299,310.275825
7,11430.790829,19.62,70.600000,5.350000,3.090000,19.59,38.629,-2.898049,18.488335,-1.357478,3.45,-7.7,21.2,0.000004,51.758794,358.109379
8,12007.048703,19.78,72.500000,5.370000,3.310000,16.32,37.538,-1.188889,18.709564,-1.740458,3.09,-8.7,22.5,0.000004,68.686869,304.485269
9,12502.021651,19.17,73.000000,5.100000,3.200000,17.27,37.138,0.526321,18.476344,-0.184948,1.24,-9.6,21.8,0.000004,48.469388,79.129692


In [6]:
lv = pd.DataFrame()
lv = df_lv.copy()
lv['EUPC_2'] = lv['EUPC'].shift(2)

In [7]:
lv = lv.dropna()

In [8]:
discretised_lv = pd.DataFrame(index=lv.index)

for col in lv.columns:
    no_unique = lv[col].nunique()
    
    if no_unique <= 1:
        discretised_lv[col] = 0
    else:
        try:
            discretised_lv[col] = pd.qcut(
                lv[col], 
                q=min(2, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_lv[col] = lv[col].rank(method='lvnse').astype(int) - 1
    num_states = discretised_lv[col].nunique()
    
    if num_states == 2:
        discretised_lv[col] = discretised_lv[col].map({0: "Low", 1: "High"})
    elif num_states == 3:
        discretised_lv[col] = discretised_lv[col].map({0: "Low", 1: "Medium", 2: "High"})
    else:
        discretised_lv[col] = discretised_lv[col].astype(str)

print("Discretised data:")
print(discretised_lv)

Discretised data:
    GDPC  VABI   EMP   QOA   QOR   YUA  REST  EBSG  GGFC  LBGD   FDI    PC  \
2    Low  High   Low  High  High  High   Low  High  High   Low   Low   Low   
3    Low  High   Low  High   Low  High   Low   Low  High   Low   Low   Low   
4    Low  High   Low   Low   Low  High   Low   Low  High   Low  High   Low   
5    Low  High   Low   Low   Low  High   Low   Low   Low  High  High   Low   
6    Low  High   Low  High   Low  High   Low   Low   Low  High  High   Low   
7    Low   Low   Low   Low   Low  High  High   Low   Low  High  High  High   
8    Low   Low   Low   Low  High   Low   Low  High   Low   Low   Low  High   
9   High   Low  High   Low  High  High   Low  High   Low  High   Low   Low   
10  High   Low  High   Low   Low   Low  High  High   Low  High  High  High   
11  High   Low  High  High  High   Low  High  High   Low  High   Low  High   
12  High   Low  High  High  High   Low  High  High  High  High  High  High   
13  High   Low  High   Low   Low   Low  High  

In [9]:
sm = StructureModel()

In [10]:
sm.add_edges_from([
    ('EUPC_2', 'FDI'),  
     ('EBSG', 'FDI'),
    ('VABI', 'FDI'),
    ('EBSG', 'PC'),
    ('VABI', 'QOA'),
    ('EUPC_2', 'CO2C'),
    ('REST', 'VABI'),
    ('EBSG', 'LBGD'),
    ('EBSG', 'FDI'),
    ('VABI', 'FDI'),
    ('LBGD', 'ARP'),
    ('YUA', 'CO2C')
    
])

In [11]:
sm.edges

OutEdgeView([('EUPC_2', 'FDI'), ('EUPC_2', 'CO2C'), ('EBSG', 'FDI'), ('EBSG', 'PC'), ('EBSG', 'LBGD'), ('VABI', 'FDI'), ('VABI', 'QOA'), ('REST', 'VABI'), ('LBGD', 'ARP'), ('YUA', 'CO2C')])

In [12]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_2.html")

Graphs/fully_connected_dk_2.html


In [13]:
bn = BayesianNetwork(sm)

bn.fit_node_states(discretised_lv)

bn.fit_cpds(discretised_lv, method="BayesianEstimator", bayes_prior="K2")

bic_baseline, ll_baseline = calculate_bic(bn, discretised_lv)
print(f"Baseline BIC: {bic_baseline:.2f} (LogLikelihood: {ll_baseline:.2f})")

Baseline BIC: 6461.94 (LogLikelihood: -3191.38)


In [14]:
bn_with_lv_6, data_with_lv_6, bic_lv_6 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['EUPC_2', 'FDI'],
    discretised_data=discretised_lv,
    raw_data=lv
)

Estimating LV based on continuous GMM distributions in: ['EUPC_2', 'FDI']
  - Removing direct edge: EUPC_2 -> FDI
LV Model BIC: 7047.47 (LogLikelihood: -3481.51)


In [15]:
bn_with_lv_7, data_with_lv_7, bic_lv_7 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['EUPC_2', 'CO2C'],
    discretised_data=discretised_lv,
    raw_data=lv
)

Estimating LV based on continuous GMM distributions in: ['EUPC_2', 'CO2C']
  - Removing direct edge: EUPC_2 -> CO2C
LV Model BIC: 7047.47 (LogLikelihood: -3481.51)
